No,２

In [2]:
import os
import shutil
import re
from pathlib import Path

# --- 共通の復元基盤 undo_utils.py を読み込む ---
import sys
from pathlib import Path

# ノートブックと同じフォルダにある undo_utils.py を import できるようにする
_here = Path.cwd()
if not (_here / "undo_utils.py").exists():
    _here = Path(globals().get("__vsc_ipynb_file__", "")).parent
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))

import undo_utils as uu

選択したメインフォルダ内のサブフォルダ内のサブサブフォルダ内にあるファイルを拡張子ごとにフォルダ分け

In [3]:
# === 【1工程目】ダイアログでのフォルダ指定と、拡張子ごとの仕分け ===
# 移動内容はすべて _undo/undo_log.json に記録され、末尾の復元セルで元に戻せます。

STEP1_NAME = "02_ファイル分別 (1工程目 仕分け)"

main_folder_path = uu.select_folder("整理したい対象のメインフォルダを選択してください")

if main_folder_path is not None:
    print("ファイルの移動を開始します...")

    # 1. 移動対象のファイルを列挙（_undo / _trash と隠しファイルは除外）
    files_to_move = list(uu.iter_files(main_folder_path))

    # 2. 拡張子ごとのフォルダへ移動
    moved = 0
    with uu.UndoJournal(main_folder_path, STEP1_NAME) as j:
        for original_file_path in files_to_move:
            file_extension = original_file_path.suffix

            if not file_extension:
                continue  # 拡張子が無いファイルは触らない

            extension_folder_name = file_extension[1:]
            relative_path = original_file_path.parent.relative_to(main_folder_path)
            main_dest_folder = main_folder_path / extension_folder_name

            if str(relative_path) == ".":
                final_dest_dir = main_dest_folder
            else:
                final_dest_dir = main_dest_folder / relative_path

            j.mkdir(final_dest_dir)
            j.move(original_file_path, final_dest_dir / original_file_path.name)
            moved += 1

    print(f"1工程目：{moved} 件のファイルの移動が完了しました。")
    print("次のセル（2工程目）を実行してください。")

選択されたフォルダ: C:/Users/0uh2j/Desktop/実験データ2026/01_本実験データ/202603-4 - 本実験圧力データ/202606・08-実験データ再々/202606・08-直接式樹脂圧力本実験データ再々/1.0～2.0mm
ファイルの移動を開始します...
復元ログを保存しました: C:\Users\0uh2j\Desktop\実験データ2026\01_本実験データ\202603-4 - 本実験圧力データ\202606・08-実験データ再々\202606・08-直接式樹脂圧力本実験データ再々\1.0～2.0mm\_undo\undo_log.json
  ステップ「02_ファイル分別 (1工程目 仕分け)」/ 操作 2832 件
1工程目：2602 件のファイルの移動が完了しました。
次のセル（2工程目）を実行してください。


不要になった空のフォルダの削除

In [4]:
# === 【2工程目】不要になった元の空フォルダの削除 ===
# rmtree による強制削除はやめました。
# 隠しファイル（.DS_Store など）が残っている場合も消さずに _trash へ退避してから
# フォルダを削除するので、復元セルで中身ごと元に戻せます。

STEP2_NAME = "02_ファイル分別 (2工程目 空フォルダ削除)"

if "main_folder_path" in globals() and main_folder_path is not None:
    print("不要になった元のサブフォルダの削除を開始します...")

    # 1工程目で作られた「拡張子フォルダ」は削除対象から外す
    extension_folders = {
        d.name for d in main_folder_path.iterdir()
        if d.is_dir() and not uu.is_reserved(d, main_folder_path)
        and "." not in d.name and d.name.startswith(".") is False
    }

    deleted = 0
    with uu.UndoJournal(main_folder_path, STEP2_NAME) as j:
        for item_path in sorted(main_folder_path.iterdir()):
            if not item_path.is_dir():
                continue
            if uu.is_reserved(item_path, main_folder_path):
                continue
            if item_path.name.startswith("."):
                continue

            # 通常のファイルが1つでも残っていれば、そのフォルダには手を触れない
            has_visible_file = any(uu.iter_files(item_path))
            if has_visible_file:
                continue

            # 深い階層から順に、隠しファイルを退避しつつフォルダを削除していく
            all_dirs = sorted(
                [item_path] + list(uu.iter_dirs(item_path, skip_hidden=False)),
                key=lambda d: len(d.parts),
                reverse=True,
            )
            for d in all_dirs:
                for leftover in sorted(d.iterdir()):
                    if leftover.is_file():
                        j.trash(leftover)  # .DS_Store などを退避（削除しない）
                if j.rmdir(d):
                    print(f"削除したフォルダ: {d}")
                    deleted += 1
                else:
                    print(f"※スキップ: {d}（中身が残っているため保持しました）")

    print(f"{deleted} 個のフォルダを削除しました。すべての処理が完了しました！")
else:
    print("エラー: メインフォルダが認識されていません。先に1工程目のセルを実行してください。")

不要になった元のサブフォルダの削除を開始します...
削除したフォルダ: C:\Users\0uh2j\Desktop\実験データ2026\01_本実験データ\202603-4 - 本実験圧力データ\202606・08-実験データ再々\202606・08-直接式樹脂圧力本実験データ再々\1.0～2.0mm\1.0mm\190℃\010
削除したフォルダ: C:\Users\0uh2j\Desktop\実験データ2026\01_本実験データ\202603-4 - 本実験圧力データ\202606・08-実験データ再々\202606・08-直接式樹脂圧力本実験データ再々\1.0～2.0mm\1.0mm\190℃\020
削除したフォルダ: C:\Users\0uh2j\Desktop\実験データ2026\01_本実験データ\202603-4 - 本実験圧力データ\202606・08-実験データ再々\202606・08-直接式樹脂圧力本実験データ再々\1.0～2.0mm\1.0mm\190℃\040
削除したフォルダ: C:\Users\0uh2j\Desktop\実験データ2026\01_本実験データ\202603-4 - 本実験圧力データ\202606・08-実験データ再々\202606・08-直接式樹脂圧力本実験データ再々\1.0～2.0mm\1.0mm\190℃\080
削除したフォルダ: C:\Users\0uh2j\Desktop\実験データ2026\01_本実験データ\202603-4 - 本実験圧力データ\202606・08-実験データ再々\202606・08-直接式樹脂圧力本実験データ再々\1.0～2.0mm\1.0mm\190℃\120
削除したフォルダ: C:\Users\0uh2j\Desktop\実験データ2026\01_本実験データ\202603-4 - 本実験圧力データ\202606・08-実験データ再々\202606・08-直接式樹脂圧力本実験データ再々\1.0～2.0mm\1.0mm\190℃\160
削除したフォルダ: C:\Users\0uh2j\Desktop\実験データ2026\01_本実験データ\202603-4 - 本実験圧力データ\202606・08-実験データ再々\202606・08-直接式樹脂圧力本実験データ再々\1.0～

ファイル分け最終段階：各拡張子フォルダがメインフォルダの直下に来るように整理

In [5]:
# === 【最終段階】各拡張子フォルダがメインフォルダの直下に来るように整理 ===

STEP3_NAME = "02_ファイル分別 (最終段階 階層の再編成)"


def reorganize_deep_folders():
    root = uu.select_folder("メインフォルダを選択してください")
    if root is None:
        return

    count = 0
    with uu.UndoJournal(root, STEP3_NAME) as j:
        # 探索の深さ: メイン(root) / サブ / サブサブ(=拡張子名) / データフォルダ
        for sub_dir in sorted(root.iterdir()):
            if not sub_dir.is_dir() or uu.is_reserved(sub_dir, root):
                continue

            for sub_sub_dir in sorted(sub_dir.iterdir()):
                if not sub_sub_dir.is_dir():
                    continue

                ext_name = sub_sub_dir.name  # サブサブフォルダ名を拡張子名として扱う

                for data_folder in sorted(sub_sub_dir.iterdir()):
                    if not data_folder.is_dir():
                        continue

                    # 新構造: メイン / 拡張子名 / 元のサブ名 / データフォルダ
                    new_parent_dir = root / ext_name / sub_dir.name
                    j.mkdir(new_parent_dir)

                    try:
                        j.move(data_folder, new_parent_dir / data_folder.name)
                        count += 1
                    except Exception as e:
                        print(f"移動エラー ({data_folder.name}): {e}")

    print(f"処理が終了しました。移動したデータフォルダ数: {count}")


reorganize_deep_folders()

選択されたフォルダ: C:/Users/0uh2j/Desktop/実験データ2026/01_本実験データ/202603-4 - 本実験圧力データ/202606・08-実験データ再々/202606・08-直接式樹脂圧力本実験データ再々/1.0～2.0mm
復元ログを保存しました: C:\Users\0uh2j\Desktop\実験データ2026\01_本実験データ\202603-4 - 本実験圧力データ\202606・08-実験データ再々\202606・08-直接式樹脂圧力本実験データ再々\1.0～2.0mm\_undo\undo_log.json
  ステップ「02_ファイル分別 (最終段階 階層の再編成)」/ 操作 27 件
処理が終了しました。移動したデータフォルダ数: 18


---
### ⏪ 復元（元に戻す）

このセルを実行すると、**このノートブックで行った直前の1工程**を巻き戻します。
（メインフォルダの `_undo/undo_log.json` に記録された履歴を使います）

繰り返し実行すれば、01〜06 のどの工程まででもさかのぼれます。
削除したファイルは `_trash` フォルダに退避されているので、これも一緒に元の場所へ戻ります。

In [6]:
# ===== 共通の復元セル =====
# 直前に実行した1工程を巻き戻します。
# 続けて実行すれば、さらに1つ前の工程へとさかのぼれます。

uu.undo_interactive()

選択されたフォルダ: C:/Users/0uh2j/Desktop/実験データ2026/01_本実験データ/202603-4 - 本実験圧力データ/202606・08-実験データ再々/202606・08-直接式樹脂圧力本実験データ再々/1.0～2.0mm
復元ログ: C:\Users\0uh2j\Desktop\実験データ2026\01_本実験データ\202603-4 - 本実験圧力データ\202606・08-実験データ再々\202606・08-直接式樹脂圧力本実験データ再々\1.0～2.0mm\_undo\undo_log.json
------------------------------------------------------------
 1. 07_末端フォルダのファイルをひとつ前のフォルダへ移動
    2026-09-01T18:52:09 / 操作 2704 件
 2. 02_ファイル分別 (1工程目 仕分け)
    2026-09-01T19:35:03 / 操作 2832 件
 3. 02_ファイル分別 (2工程目 空フォルダ削除)
    2026-09-01T19:36:22 / 操作 114 件
 4. 02_ファイル分別 (最終段階 階層の再編成)
    2026-09-01T19:37:36 / 操作 27 件 ← 次に復元されるステップ
------------------------------------------------------------

復元中: 02_ファイル分別 (最終段階 階層の再編成)（2026-09-01T19:37:36）
  完了: 27 件を復元

復元が完了しました。
